# Running SIMBA-POP: a farm simulation, end to end

load parameters -> load weather -> load field observations (ribbon-tag production and farm-wide
sucker selection) -> load the initial population -> construct the engine -> run -> look at the
output.

`SimbaPopEngine` is the **only** entry point: a one-row seed and a full field census are the same
`initial_population` argument, and each input file is read by exactly one dedicated, standardized
reader class (`py_simba_pop.readers`), regardless of which farm the data came from.

Two of the inputs -- **ribbon-tag production** and **farm-wide sucker selection** -- are
**optional**. Provide them and the engine seeds its chains with your real counts (bunches bagged,
followers selected); omit either (or both) and the engine falls back to Tixier et al. (2004)'s own
mechanistic prediction for whatever is missing -- **always with an explicit notice, never
silently**. Harvest is always the model's own prediction. The committed run uses `data/synthetic`
(an established 165-ha synthetic farm) with both inputs; its desuckering records have a two-week
gap near the end, so section 6 shows what the notices look like. Set the `SIMBA_POP_DATA_DIR`
environment variable to run the same notebook on another farm directory.

In [33]:
from pathlib import Path

# This notebook lives in docs/examples/: run it from there so data/ and utils/ resolve
# (py_simba_pop itself comes from `pip install py-simba-pop`).

In [34]:
import os
from collections import Counter

import pandas as pd
import plotly.graph_objects as go

from py_simba_pop import (
    UNASSIGNED_TAG,
    ParameterSet,
    PopulationInventoryReader,
    RibbonProductionData,
    RibbonProductionReader,
    SimbaPopEngine,
    SuckerSelectionData,
    SuckerSelectionReader,
    WeatherFileReader,
    WeatherProvider,
)

# Default: the committed synthetic farm. Set SIMBA_POP_DATA_DIR (e.g. data/<farm_name>) for another farm.
DATA_DIR = Path(os.environ.get("SIMBA_POP_DATA_DIR", "data/synthetic"))
print(f"DATA_DIR = {DATA_DIR.as_posix()}")

# Plot colours for the ribbon tags of the bundled datasets (any other tag gets a plotly default).
RIBBON_COLOURS = {
    "Amarillo": "#eda100",
    "Azul": "#2a78d6",
    "Blanco": "#b4b2a9",
    "Café": "#8b5a2b",
    "Morado": "#7b3fb5",
    "Negro": "#2b2b2b",
    "Rojo": "#e34948",
    "Verde": "#008300",
    UNASSIGNED_TAG: "#898781",
}

DATA_DIR = data/synthetic


## 1. Parameters

Every parameter has a hardcoded Tixier et al. (2004) default; `utils/build_custom_parameters.py`
fits your own harvest-timing parameters from ribbon-tagged records and writes a YAML file that
`ParameterSet.from_yaml` loads. `general.rate_basis` is `"cohort"` by default: each
flowering / sucker-selection / harvest curve is a per-cohort distribution that sums to 1, so every
plant flowers once, is harvested once and keeps exactly one follower -- a farm's number of
production units (mats) stays constant without any carrying capacity. The legacy reading is
`params.override("general", {"rate_basis": "current_pool"})`.

In [35]:
params = ParameterSet.from_tixier_defaults()
display(params)
print(f"rate_basis = {params.general['rate_basis']!r}")

ParameterSet(general={'base_temperature_c': 14.0, 'week_length_days': 7, 'population_epsilon': 0.5, 'max_age_weeks_vegetative': 150, 'max_age_weeks_reproductive': 20, 'rate_basis': 'cohort'}, flowering={'threshold_dd': 2350.0, 'amplitude': 0.12, 'peak_weeks': 6.35, 'spread': 0.47}, sucker_selection={'threshold_dd': 2180.0, 'amplitude': 1.0, 'peak_weeks': 3.25, 'spread': 0.31, 'carrying_capacity': None}, harvest={'threshold_dd': 900.0, 'hazard_weeks': [0.2, 0.625, 1.0]}, mortality_replacement={'death_rate_vegetative': 0.0, 'death_rate_reproductive': 0.0, 'replacement_rate': 0.0}, source=('tixier_defaults',))

rate_basis = 'cohort'


## 2. Weather

The model's only paper-native input is mean daily temperature. `WeatherProvider.from_file_and_params`
reads a `.csv`/`.xlsx` file via the one dedicated `WeatherFileReader` and wires in
`params.general.base_temperature_c`/`week_length_days` -- no other weather column is read.

In [36]:
result = WeatherFileReader.read(DATA_DIR / "weather.csv")
weather_df = result.data

print(weather_df.shape)
display(weather_df)

(3640, 2)


,date,tmean
0,2016-01-04,24.964932
1,2016-01-05,25.651860
2,2016-01-06,25.644048
3,2016-01-07,25.021183
4,2016-01-08,25.289745
...,...,...
3635,2025-12-17,24.571699
3636,2025-12-18,24.843750
3637,2025-12-19,24.998004
3638,2025-12-20,24.446231


In [37]:
# Visualizing the raw weather data tmean at daily timestamp
fig = go.Figure()
fig.add_trace(go.Scatter(x=weather_df["date"], y=weather_df["tmean"].round(2), name="tmean", line=dict(color="#FFC107", width=2)))

fig.update_layout(
    title=f"{DATA_DIR.name}: tmean over time",
    xaxis_title="Date",
    yaxis_title="tmean (°C)",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [38]:
weather = WeatherProvider.from_file_and_params(DATA_DIR / "weather.csv", params)
print(
    f"Weather covers week_index {weather.min_week_index}..{weather.max_week_index} "
    f"({weather.calendar_date_for_week(weather.min_week_index)} .. {weather.calendar_date_for_week(weather.max_week_index)})"
)

# The derived weekly values: thermal increment (degree-days above the base temperature) and its running total
weeks = list(range(weather.min_week_index, weather.max_week_index + 1))
weather_weekly_df = pd.DataFrame(
    {
        "week_index": weeks,
        "date": [weather.calendar_date_for_week(w) for w in weeks],
        "thermal_increment": [weather.thermal_increment(w) for w in weeks],
        "thermal_trajectory": weather.thermal_trajectory(weather.min_week_index, weather.max_week_index),
    }
)
display(weather_weekly_df)

Weather covers week_index 0..519 (2016-01-04 .. 2025-12-15)


,week_index,date,thermal_increment,thermal_trajectory
0,0,2016-01-04,77.942012,77.942012
1,1,2016-01-11,70.909447,148.851459
2,2,2016-01-18,80.541666,229.393125
3,3,2016-01-25,76.353892,305.747017
4,4,2016-02-01,74.844415,380.591432
...,...,...,...,...
515,515,2025-11-17,64.709531,41955.853167
516,516,2025-11-24,75.354300,42031.207467
517,517,2025-12-01,74.116227,42105.323694
518,518,2025-12-08,77.830281,42183.153975


In [39]:
# Visualizing the derived weekly thermal increment (degree-days)
fig = go.Figure()
fig.add_trace(go.Scatter(x=weather_weekly_df["date"], y=weather_weekly_df["thermal_increment"], name="thermal_increment", line=dict(color="#FFC107", width=2)))

fig.update_layout(
    title=f"{DATA_DIR.name}: thermal_increment over time",
    xaxis_title="Date",
    yaxis_title="Degree-days per week (°C·d)",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

## 3. Field observations (ribbon-tag production and farm-wide sucker selection) [optional]

`production.csv` -- one row per (ribbon tag, date): bunches bagged and cut, the age at cut (weeks
since bagging) and, when recorded, field losses and packing-plant rejects -- and
`sucker_selection.csv` -- one row per date: followers selected farm-wide, split by whether the
mother had already flowered (parous) or not (nulliparous) -- are each read by their own
standardized reader. A reader reports any optional column it did not find as a `FEATURE_DISABLED`
notice (printed below); gaps in weekly **coverage** are only known once the engine steps through
the calendar (section 6).

In [40]:
production_result = RibbonProductionReader.read(DATA_DIR / "production.csv")
for n in production_result.notices:
    print(f"[{n.kind.value}] {n.subject}: {n.message}")

production_df = production_result.data.assign(
    week_index=lambda df: df["date"].map(weather.week_index_for_date)
)
display(production_df)

ribbon_production = RibbonProductionData(production_result.data, weather)
PRODUCTION_FIRST_WEEK = int(production_df["week_index"].min())
PRODUCTION_LAST_WEEK = int(production_df["week_index"].max())
print(f"Ribbon tags: {ribbon_production.tags()}")
print(
    f"Production records cover week_index {PRODUCTION_FIRST_WEEK}..{PRODUCTION_LAST_WEEK} "
    f"({weather.calendar_date_for_week(PRODUCTION_FIRST_WEEK)} .. {weather.calendar_date_for_week(PRODUCTION_LAST_WEEK)})"
)

[feature_disabled] processed: 'processed' column not present in WindowsPath('data/synthetic/production.csv'); this is a non-paper extension with no Tixier equation to estimate it from, so it will simply not be modeled (treated as absent, not zero).


,tag,date,bagged,cut,age_at_cut,field_losses,rejected,processed,week_index
0,Amarillo,2018-12-31,NaN,NaN,NaN,24.0,NaN,NaN,156
1,Azul,2018-12-31,NaN,1425.0,12.0,20.0,33.0,NaN,156
2,Blanco,2018-12-31,NaN,NaN,NaN,29.0,NaN,NaN,156
3,Café,2018-12-31,NaN,NaN,NaN,4.0,NaN,NaN,156
4,Morado,2018-12-31,NaN,NaN,NaN,29.0,NaN,NaN,156
...,...,...,...,...,...,...,...,...,...
2874,Café,2025-11-24,NaN,NaN,NaN,6.0,NaN,NaN,516
2875,Morado,2025-11-24,NaN,NaN,NaN,36.0,NaN,NaN,516
2876,Negro,2025-11-24,NaN,1511.0,14.0,3.0,28.0,NaN,516
2877,Rojo,2025-11-24,8115.0,NaN,NaN,25.0,NaN,NaN,516


Ribbon tags: ('Amarillo', 'Azul', 'Blanco', 'Café', 'Morado', 'Negro', 'Rojo', 'Verde')
Production records cover week_index 156..516 (2018-12-31 .. 2025-11-24)


In [41]:
# Bunches bagged each week, by ribbon colour
fig = go.Figure()
for tag in ribbon_production.tags():
    series = production_df[(production_df["tag"] == tag) & (production_df["bagged"] > 0)]
    fig.add_trace(go.Bar(x=series["date"], y=series["bagged"], name=tag, marker_color=RIBBON_COLOURS.get(tag)))
fig.update_layout(
    barmode="stack",
    bargap=0.1,
    title=f"{DATA_DIR.name}: bunches bagged per week, by ribbon colour",
    xaxis_title="Date",
    yaxis_title="Bunches bagged",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [42]:
# Farm-wide weekly totals -- sum the numeric count columns only (never the string `tag` column)
COUNT_COLUMNS = ["bagged", "cut", "field_losses", "rejected"]
production_weekly = production_df.groupby("week_index")[COUNT_COLUMNS].sum(min_count=1).reset_index()
production_weekly.insert(1, "date", production_weekly["week_index"].map(weather.calendar_date_for_week))
display(production_weekly)

,week_index,date,bagged,cut,field_losses,rejected
0,156,2018-12-31,8464.0,7581.0,136.0,188.0
1,157,2019-01-07,8742.0,6057.0,124.0,145.0
2,158,2019-01-14,8742.0,3988.0,145.0,105.0
3,159,2019-01-21,8543.0,9932.0,137.0,279.0
4,160,2019-01-28,8359.0,7962.0,151.0,172.0
...,...,...,...,...,...,...
356,512,2025-10-27,8898.0,9049.0,146.0,222.0
357,513,2025-11-03,8738.0,8966.0,173.0,241.0
358,514,2025-11-10,8451.0,8856.0,176.0,212.0
359,515,2025-11-17,8221.0,7020.0,173.0,184.0


In [43]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=production_weekly["date"], y=production_weekly["bagged"], name="bagged", line=dict(color="#FFC107", width=2)))
fig.add_trace(go.Scatter(x=production_weekly["date"], y=production_weekly["cut"], name="cut", line=dict(color="#1FAC42", width=2)))
fig.add_trace(go.Scatter(x=production_weekly["date"], y=production_weekly["field_losses"], name="field_losses", line=dict(color="#FD3939", width=2)))
fig.add_trace(go.Scatter(x=production_weekly["date"], y=production_weekly["rejected"], name="rejected (part of cut)", line=dict(color="#FF8902", width=2)))

fig.update_layout(
    title=f"{DATA_DIR.name}: farm-wide weekly production records",
    xaxis_title="Date",
    yaxis_title="Bunches",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [44]:
# sucker_selection.csv is optional: without it the engine uses its own mechanistic Eq 3 follower prediction
if (DATA_DIR / "sucker_selection.csv").exists():
    sucker_selection_result = SuckerSelectionReader.read(DATA_DIR / "sucker_selection.csv")
    for n in sucker_selection_result.notices:
        print(f"[{n.kind.value}] {n.subject}: {n.message}")
    sucker_selection_df = sucker_selection_result.data
    display(sucker_selection_df)
    sucker_selection = SuckerSelectionData(sucker_selection_df, weather)
else:
    sucker_selection_df = None
    sucker_selection = None
    print(f"No sucker_selection.csv in {DATA_DIR.as_posix()}; the engine will predict follower selection itself.")

,date,parous,nulliparous,total
0,2018-12-31,686.0,8168.0,8854.0
1,2019-01-07,738.0,8349.0,9087.0
2,2019-01-14,796.0,8459.0,9255.0
3,2019-01-21,815.0,8547.0,9362.0
4,2019-01-28,791.0,8285.0,9076.0
...,...,...,...,...
354,2025-10-27,953.0,8334.0,9287.0
355,2025-11-03,961.0,7869.0,8830.0
356,2025-11-10,931.0,7108.0,8039.0
357,2025-11-17,868.0,6586.0,7454.0


In [45]:
if sucker_selection_df is not None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sucker_selection_df["date"], y=sucker_selection_df["total"], name="total", line=dict(color="#1F6D4C", width=2)))
    fig.add_trace(go.Scatter(x=sucker_selection_df["date"], y=sucker_selection_df["nulliparous"], name="nulliparous (mother not yet flowered)", line=dict(color="#2a78d6", width=2)))
    fig.add_trace(go.Scatter(x=sucker_selection_df["date"], y=sucker_selection_df["parous"], name="parous (mother already flowered)", line=dict(color="#eb6834", width=2)))
    fig.update_layout(
        title=f"{DATA_DIR.name}: followers selected per week",
        xaxis_title="Date",
        yaxis_title="Followers selected",
        template="plotly_white",
        legend=dict(orientation="h", y=1.15),
    )
    fig.show()
else:
    print("No sucker-selection records to plot.")

## 4. Initial population and the engine

`initial_population.csv` is the field census the simulation starts from, one row per cohort:
pre-flowering (vegetative) plants by weeks since they were selected or planted, and post-flowering
plants by weeks since their bunch was bagged and the ribbon `tag` that bunch carries. The same
`initial_population` argument takes a one-row seed, a census like this one, or an
`engine.snapshot_cohorts()` inventory -- there is no separate cold-start constructor.

`PopulationInventoryReader` ignores the file's `snapshot_date` column, so we read it from the raw
csv and convert it to the engine's `start_week_index`: the census describes the farm at the
*start* of that week.

In [46]:
initial_population = PopulationInventoryReader.read(DATA_DIR / "initial_population.csv").data
snapshot_date = pd.to_datetime(pd.read_csv(DATA_DIR / "initial_population.csv")["snapshot_date"].iloc[0]).date()
START_WEEK_INDEX = weather.week_index_for_date(snapshot_date)
print(
    f"Census snapshot_date {snapshot_date} -> START_WEEK_INDEX = {START_WEEK_INDEX} "
    f"(week starting {weather.calendar_date_for_week(START_WEEK_INDEX)})"
)

census_by_chain = initial_population.groupby("chain").agg(
    rows=("population", "size"),
    plants=("population", "sum"),
    min_age_weeks=("age_weeks", "min"),
    max_age_weeks=("age_weeks", "max"),
)
display(census_by_chain)
print(f"Standing plants in the census: {initial_population['population'].sum():,.0f}")

post_census = initial_population[initial_population["chain"] == "post_flowering"]
if not post_census.empty and "tag" in post_census.columns:
    display(post_census.groupby("tag")["population"].sum().rename("bunches on the plant").to_frame().T)

Census snapshot_date 2018-12-31 -> START_WEEK_INDEX = 156 (week starting 2018-12-31)


,rows,plants,min_age_weeks,max_age_weeks
chain,,,,
post_flowering,14,94820.0,1,14
pre_flowering,78,322476.0,1,78


Standing plants in the census: 417,296


tag,Amarillo,Azul,Blanco,Café,Morado,Negro,Rojo,Verde
bunches on the plant,14630.0,14311.0,14514.0,7212.0,14952.0,8738.0,7189.0,13274.0


In [47]:
# Census age distribution (pre-flowering: weeks since selection/planting; post-flowering: weeks since bagging)
fig = go.Figure()
for chain, colour in [("pre_flowering", "#2a78d6"), ("post_flowering", "#eb6834")]:
    by_age = initial_population[initial_population["chain"] == chain].groupby("age_weeks")["population"].sum()
    fig.add_trace(go.Bar(x=by_age.index, y=by_age.values, name=chain, marker_color=colour))
fig.update_layout(
    barmode="group",
    title=f"{DATA_DIR.name}: initial population by age ({snapshot_date})",
    xaxis_title="age_weeks",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [48]:
ENGINE_KWARGS = dict(
    start_week_index=START_WEEK_INDEX,
    ribbon_production=ribbon_production,
    sucker_selection=sucker_selection,  # None -> the mechanistic Eq 3 inflow every week, with a notice
    tags=list(ribbon_production.tags()),
)
engine = SimbaPopEngine(params, weather, initial_population, **ENGINE_KWARGS)

post_at_start = sum(chain.total_population() for chain in engine.post_chains.values())
print(f"Engine clock starts at week_index {engine.current_week_index}")
print(f"Standing at construction: {engine.pre_chain.total_population():,.0f} pre-flowering, {post_at_start:,.0f} post-flowering")
print(f"Construction-time notices: {[n.subject for n in engine.notices]}")

Engine clock starts at week_index 156
Standing at construction: 322,476 pre-flowering, 94,820 post-flowering
Construction-time notices: []


## 5. Run

`py_simba_pop` is strictly weather driven, so a run can only cover weeks with weather data. This one
runs from `START_WEEK_INDEX` (the census week) through `LAST_USABLE_WEEK`, the last week with both
weather and production records, inclusive. On weeks with ribbon records the post-flowering chains
are seeded with the real bagged counts; on weeks with desuckering records the pre-flowering chain
is seeded with the real number of followers selected. Flowering of the vegetative plants and the
harvest of every bunch are always the model's own prediction.

In [49]:
LAST_USABLE_WEEK = min(weather.max_week_index, PRODUCTION_LAST_WEEK)
N_WEEKS = LAST_USABLE_WEEK - START_WEEK_INDEX + 1
print(
    f"Running engine for {N_WEEKS} weeks, week_index {START_WEEK_INDEX}..{LAST_USABLE_WEEK}, starting from "
    f"{weather.calendar_date_for_week(START_WEEK_INDEX)} (last week starts {weather.calendar_date_for_week(LAST_USABLE_WEEK)})"
)

output = engine.run(N_WEEKS)
display(output)

Running engine for 361 weeks, week_index 156..516, starting from 2018-12-31 (last week starts 2025-11-24)


,week_index,date,Nt,Nt_pre,Nt_post,Ht,FLt,St,sucker_output_pre,sucker_output_post,Dt,sucker_input_used,flowering_routed_to_unassigned,notice_count,pruned_population_this_week
0,156,2018-12-31,418302.715354,322794.877854,95507.837500,7640.162500,8535.122146,9919.631901,9853.596071,66.035831,0.0,8854.0,0.0,0,0.000000
1,157,2019-01-07,421333.038447,323075.255406,98257.783041,5868.054459,8806.622448,9598.685630,9528.431089,70.254541,0.0,9087.0,0.0,0,0.000000
2,158,2019-01-14,426324.768662,323525.685082,102799.083580,4055.699461,8804.570324,8365.140652,8301.782379,63.358273,0.0,9255.0,0.0,0,0.000000
3,159,2019-01-21,425448.572143,324278.234625,101170.337518,10034.746062,8606.555966,6022.731503,5977.272073,45.459431,0.0,9362.0,0.0,0,2.894492
4,160,2019-01-28,426261.492909,324934.919504,101326.573406,8051.764112,8419.315121,6892.519744,6860.693691,31.826053,0.0,9076.0,0.0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,512,2025-10-27,437360.325698,322328.737724,115031.587974,9034.011096,8897.737570,8375.149111,8308.230466,66.918645,0.0,9287.0,0.0,0,0.432175
357,513,2025-11-03,437076.130811,322420.062083,114656.068728,8940.519246,8738.228478,5867.518018,5807.462960,60.055058,0.0,8830.0,0.0,0,0.447163
358,514,2025-11-10,436094.214343,322007.503444,114086.710899,8844.357829,8451.081261,6194.000701,6150.802546,43.198154,0.0,8039.0,0.0,0,0.477378
359,515,2025-11-17,436336.020059,321239.697188,115096.322871,7038.388028,8221.310179,7322.819194,7292.218151,30.601043,0.0,7454.0,0.0,0,0.496076


> `output` and `tag_output` (below) are not two views of the same population.
>
> - `output` is the farm-wide summary, one row per `week_index`: `Nt = Nt_pre + Nt_post` covers the
>   pre-flowering chain and every post-flowering chain; `Ht` is the harvest of all post-flowering
>   chains; `FLt` is the model's flowering prediction; `St` its follower-selection prediction
>   (`sucker_output_pre` = followers of not-yet-flowered mothers, `sucker_output_post` = followers of
>   already-flowered mothers); `sucker_input_used` is what actually seeded the pre-flowering chain.
> - `tag_output` is the per-tag breakdown of the post-flowering chains only, one row per
>   (`week_index`, `tag`), including the reserved `__unassigned__` chain: `Nt`, `Ht`, `Xt` (field
>   losses removed; rejected bunches are part of the cut, not removed again) and `bagged_input`
>   (what seeded that tag's new cohort).
>
> So `output["Nt_post"]` equals the weekly sum of `tag_output["Nt"]`, and `output["Ht"]` the weekly
> sum of `tag_output["Ht"]`. See the field definitions in the
> [quick start](../quickstart.rst).

In [50]:
# Per-tag output data from the engine
tag_output = engine.get_tag_output()
display(tag_output)

,week_index,date,tag,Nt,Ht,Xt,bagged_input,cut_observed,age_at_cut_observed
0,156,2018-12-31,Amarillo,14606.000000,0.000000,24.0,0.0,NaN,NaN
1,156,2018-12-31,Azul,12836.400000,1454.600000,20.0,0.0,1425.0,12.0
2,156,2018-12-31,Blanco,14485.000000,0.000000,29.0,0.0,NaN,NaN
3,156,2018-12-31,Café,7208.000000,0.000000,4.0,0.0,NaN,NaN
4,156,2018-12-31,Morado,14923.000000,0.000000,29.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3244,516,2025-11-24,Morado,17042.845035,0.000000,36.0,0.0,NaN,NaN
3245,516,2025-11-24,Negro,8748.379624,1537.309072,3.0,0.0,1511.0,14.0
3246,516,2025-11-24,Rojo,16908.699299,0.000000,25.0,8115.0,NaN,NaN
3247,516,2025-11-24,Verde,15785.767706,1753.842457,25.0,0.0,1739.0,13.0


### Flowering: the model's prediction vs. the recorded bagging

On every week covered by ribbon records the engine seeds the post-flowering chains with the **real**
bagged counts (`tag_output["bagged_input"]`). `output["FLt"]` is a different quantity: the model's
own flowering prediction from the pre-flowering chain, computed every week whether or not real
bagging exists (on a week without ribbon coverage it is what seeds `__unassigned__`). Comparing
the two checks the pre-flowering chain: do the plants the model says flower this week match the
bunches the farm actually bagged? On `data/synthetic`, where real desuckering records drive the
pre-flowering chain, they agree almost exactly: r2 = 0.9996 over the 361 simulated weeks (see
`03_validating_farm_simulation.ipynb` for runs without those records).

In [51]:
seeded = (
    tag_output[tag_output["tag"] != UNASSIGNED_TAG]
    .groupby("week_index", as_index=False)["bagged_input"]
    .sum()
)
flowering = (
    output[["week_index", "date", "FLt"]]
    .merge(production_weekly[["week_index", "bagged"]], on="week_index", how="left")
    .merge(seeded, on="week_index", how="left")
)
# On a covered week the engine treats an unreported bagging count as zero, so do the same here.
covered = flowering[flowering["week_index"].map(ribbon_production.week_is_covered)].fillna({"bagged": 0.0})

residual = covered["bagged"] - covered["FLt"]
r2 = 1.0 - (residual**2).sum() / ((covered["bagged"] - covered["bagged"].mean()) ** 2).sum()
driver = "real sucker-selection records" if sucker_selection is not None else "the mechanistic Eq 3 prediction"
print(f"{len(covered)} of {len(flowering)} simulated weeks are covered by ribbon records")
print(f"Largest |bagged_input seeded - bagged recorded| on those weeks: {(covered['bagged_input'] - covered['bagged']).abs().max():.1e}")
print(f"Recorded bagging: {covered['bagged'].sum():,.0f} bunches; model FLt: {covered['FLt'].sum():,.0f}")
print(f"r2 of FLt against recorded weekly bagging: {r2:.4f} (pre-flowering inflow from {driver})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=flowering["date"], y=flowering["bagged"], name="bagged (recorded)", line=dict(color="#2b2b2b", width=2)))
fig.add_trace(go.Scatter(x=flowering["date"], y=flowering["FLt"].round(1), name="FLt (model flowering)", line=dict(color="#FFC107", width=2, dash="dot")))
fig.update_layout(
    title=f"{DATA_DIR.name}: model flowering vs. recorded bagging (r2 = {r2:.4f})",
    xaxis_title="Date",
    yaxis_title="Bunches per week",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

361 of 361 simulated weeks are covered by ribbon records
Largest |bagged_input seeded - bagged recorded| on those weeks: 0.0e+00
Recorded bagging: 3,127,734 bunches; model FLt: 3,127,844
r2 of FLt against recorded weekly bagging: 0.9996 (pre-flowering inflow from real sucker-selection records)


In [52]:
# Farm-wide Ht is exactly the sum of the per-tag harvests (tag_output covers every post-flowering chain)
total_harvested_farm = output["Ht"].sum()
total_harvested_tags = tag_output["Ht"].sum()
print(f"Total harvested from engine output: {total_harvested_farm:,.2f}")
print(f"Total harvested from tag_output:    {total_harvested_tags:,.2f}")
print(f"Difference: {total_harvested_farm - total_harvested_tags:.2e}")

post_by_week = tag_output.groupby("week_index")["Nt"].sum()
print(f"Largest weekly |Nt_post - sum of tag Nt|: {(output.set_index('week_index')['Nt_post'] - post_by_week).abs().max():.2e}")

Total harvested from engine output: 3,049,899.83
Total harvested from tag_output:    3,049,899.83
Difference: 0.00e+00
Largest weekly |Nt_post - sum of tag Nt|: 1.46e-11


## 6. Notices

Every week the real data had a gap, the engine logged exactly why and what it used instead.
`engine.notices` is the complete, structured record -- each `Notice` has `kind`, `subject`,
`message`, `source`, `week_index` and `tag`. Print it yourself, as below, rather than relying on
`emit_warnings()`, whose stderr output can be deduplicated by Python's warning filter.

In [53]:
print(f"{len(engine.notices)} notices total")
notice_counts = Counter((n.subject, n.kind.value) for n in engine.notices)
display(
    pd.DataFrame(
        [{"subject": subject, "kind": kind, "count": count} for (subject, kind), count in sorted(notice_counts.items())],
        columns=["subject", "kind", "count"],
    )
)

6 notices total


,subject,kind,count
0,field_losses,feature_disabled,4
1,sucker_selection,fallback_estimate,2


In [54]:
for subject in sorted({n.subject for n in engine.notices}):
    examples = [n for n in engine.notices if n.subject == subject]
    print(f"{subject}: {len(examples)} notice(s), e.g.")
    for n in examples[:3]:
        when = "construction" if n.week_index is None else f"week_index {n.week_index} ({weather.calendar_date_for_week(n.week_index)})"
        where = "" if n.tag is None else f", tag {n.tag}"
        print(f"  [{n.kind.value}] {when}{where}: {n.message}")

field_losses: 4 notice(s), e.g.
  [feature_disabled] week_index 285 (2021-06-21), tag Azul: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
  [feature_disabled] week_index 319 (2022-02-14), tag Blanco: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
  [feature_disabled] week_index 331 (2022-05-09), tag Negro: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
sucker_selection: 2 notice(s), e.g.
  [fallback_estimate] week_index 508 (2025-09-29): no observed farm-wide sucker-selection total for this week; using the mechanistic Eq 3 pooled-sucker prediction (9187.0) instead.
  [fallback_estimate] week_index 509 (2025-10-06): no observed farm-wide sucker-selection total for this week; using the mechanist

## 7. Output

The standing population and the weekly flows, on separate charts since they differ in scale by
well over an order of magnitude:

In [55]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=output["date"], y=output["Nt"].round(1), name="Nt (total population)", line=dict(color="#2E7D32", width=2)))
fig.add_trace(go.Scatter(x=output["date"], y=output["Nt_pre"].round(1), name="Nt_pre (pre-flowering)", line=dict(color="#2a78d6", width=2)))
fig.add_trace(go.Scatter(x=output["date"], y=output["Nt_post"].round(1), name="Nt_post (post-flowering)", line=dict(color="#eb6834", width=2)))

fig.update_layout(
    title=f"{DATA_DIR.name}: standing population over time",
    xaxis_title="Date",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [56]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=output["date"], y=output["Ht"].round(1), name="Ht (harvested)", line=dict(color="#1565C0", width=2)))
fig.add_trace(go.Scatter(x=output["date"], y=output["FLt"].round(1), name="FLt (flowering)", line=dict(color="#FFC107", width=2)))
fig.add_trace(go.Scatter(x=output["date"], y=output["St"].round(1), name="St (followers, model)", line=dict(color="#2FC49A", width=2)))
fig.add_trace(go.Scatter(x=output["date"], y=output["sucker_input_used"].round(1), name="sucker_input_used (seeded)", line=dict(color="#1F6D4C", width=2, dash="dot")))

fig.update_layout(
    title=f"{DATA_DIR.name}: weekly flowering, harvest and follower selection",
    xaxis_title="Date",
    yaxis_title="Plants per week",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

Per-tag inflow (real bagging, or the reserved `__unassigned__` chain on weeks without ribbon
coverage) and per-tag harvest:

In [57]:
# Bagging input (flowering) by tag over time
fig = go.Figure()
for tag in [*ribbon_production.tags(), UNASSIGNED_TAG]:
    series = tag_output[tag_output["tag"] == tag]
    fig.add_trace(go.Scatter(x=series["date"], y=series["bagged_input"].round(1), name=tag, mode="lines", line=dict(color=RIBBON_COLOURS.get(tag))))
fig.update_layout(
    title=f"{DATA_DIR.name}: weekly inflow by tag ({UNASSIGNED_TAG!r} = no real ribbon coverage that week)",
    xaxis_title="Date",
    yaxis_title="Bunches bagged",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [58]:
# Harvested output (Ht) by tag over time
fig = go.Figure()
for tag in [*ribbon_production.tags(), UNASSIGNED_TAG]:
    series = tag_output[tag_output["tag"] == tag]
    fig.add_trace(go.Scatter(x=series["date"], y=series["Ht"].round(1), name=tag, mode="lines", line=dict(color=RIBBON_COLOURS.get(tag))))
fig.update_layout(
    title=f"{DATA_DIR.name}: weekly harvest by tag ({UNASSIGNED_TAG!r} = no real ribbon coverage that week)",
    xaxis_title="Date",
    yaxis_title="Bunches harvested",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

## 8. Population by stage and age over time

`output` only carries chain totals. For the full age structure, step a fresh engine (built exactly
like `engine`) one week at a time and take `snapshot_cohorts()` at the start of every week -- the
same format as the census, so the first snapshot is `initial_population` itself. Ages are weeks
since selection/planting on the pre-flowering chain and weeks since bagging on the post-flowering
chain; a cohort added during week *w* is 1 week old in the snapshot at the start of week *w* + 1, so
the youngest age in any snapshot is 1.

In [59]:
PRE_BANDS = [(0, 9), (10, 19), (20, 29), (30, 39), (40, None)]  # weeks since selection/planting
POST_BANDS = [(0, 4), (5, 9), (10, 14), (15, None)]  # weeks since bagging
SNAPSHOT_WEEKS = list(range(START_WEEK_INDEX, START_WEEK_INDEX + N_WEEKS))
SNAPSHOT_DATES = [weather.calendar_date_for_week(w) for w in SNAPSHOT_WEEKS]

stepped = SimbaPopEngine(params, weather, initial_population, **ENGINE_KWARGS)
snapshots = []
for _ in SNAPSHOT_WEEKS:
    snapshot = stepped.snapshot_cohorts()
    snapshots.append(snapshot[["chain", "age_weeks", "population"]].assign(week_index=stepped.current_week_index))
    stepped.step()
cohort_history = pd.concat(snapshots, ignore_index=True)
print(f"Stepping reproduces engine.run(): {stepped.get_output().equals(output)}")
print(f"{len(cohort_history):,} cohort rows over {len(SNAPSHOT_WEEKS)} weekly snapshots")


def band_label(lo, hi):
    return f"{lo}+ wk" if hi is None else f"{lo}-{hi} wk"


def population_by_band(chain, bands):
    # Population per (snapshot week, age band) for one chain -- weeks as rows, bands as columns
    rows = cohort_history[cohort_history["chain"] == chain]
    labels = [band_label(lo, hi) for lo, hi in bands]
    if rows.empty:
        return pd.DataFrame(0.0, index=SNAPSHOT_WEEKS, columns=labels)
    band = pd.cut(rows["age_weeks"], bins=[lo for lo, _ in bands] + [float("inf")], labels=labels, right=False)
    table = rows.groupby([rows["week_index"], band], observed=False)["population"].sum().unstack(fill_value=0.0)
    return table.reindex(index=SNAPSHOT_WEEKS, columns=labels, fill_value=0.0)


pre_bands = population_by_band("pre_flowering", PRE_BANDS)
post_bands = population_by_band("post_flowering", POST_BANDS)
display(
    pd.concat({"pre_flowering": pre_bands.mean(), "post_flowering": post_bands.mean()})
    .rename("mean standing plants per snapshot")
    .round(0)
    .to_frame()
)
for chain, table in [("pre_flowering", pre_bands), ("post_flowering", post_bands)]:
    total = table.sum(axis=1)
    print(f"{chain}: {total.min():,.0f} .. {total.max():,.0f} standing (first snapshot {total.iloc[0]:,.0f}, last {total.iloc[-1]:,.0f})")
print(
    f"Over the run: {output['sucker_input_used'].sum():,.0f} followers seeded, {output['FLt'].sum():,.0f} plants flowered, "
    f"{output['Ht'].sum():,.0f} bunches harvested"
)

Stepping reproduces engine.run(): True
32,672 cohort rows over 361 weekly snapshots


mean standing plants per snapshot
               age_weeks                                   
pre_flowering  0-9 wk                               77938.0
               10-19 wk                             86236.0
               20-29 wk                             86074.0
               30-39 wk                             62299.0
               40+ wk                                9684.0
post_flowering 0-4 wk                               34554.0
               5-9 wk                               42863.0
               10-14 wk                             30044.0
               15+ wk                                  69.0

pre_flowering: 317,800 .. 329,185 standing (first snapshot 322,476, last 321,240)
post_flowering: 94,521 .. 119,313 standing (first snapshot 94,820, last 115,096)
Over the run: 3,125,927 followers seeded, 3,127,844 plants flowered, 3,049,900 bunches harvested


In [60]:
PRE_COLOURS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]  # young -> old
fig = go.Figure()
for label, colour in zip(pre_bands.columns, PRE_COLOURS):
    fig.add_trace(
        go.Scatter(x=SNAPSHOT_DATES, y=pre_bands[label].round(0), name=label, mode="lines", stackgroup="pre",
                   line=dict(color=colour, width=0.5), fillcolor=colour)
    )
fig.update_layout(
    title=f"{DATA_DIR.name}: pre-flowering plants by weeks since selection/planting",
    xaxis_title="Date",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [61]:
POST_COLOURS = ["#f5b08a", "#eb6834", "#b8461a", "#7a2c0c"]  # just bagged -> oldest
fig = go.Figure()
for label, colour in zip(post_bands.columns, POST_COLOURS):
    fig.add_trace(
        go.Scatter(x=SNAPSHOT_DATES, y=post_bands[label].round(0), name=label, mode="lines", stackgroup="post",
                   line=dict(color=colour, width=0.5), fillcolor=colour)
    )
fig.update_layout(
    title=f"{DATA_DIR.name}: bunches on the plant by weeks since bagging",
    xaxis_title="Date",
    yaxis_title="Bunches",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [62]:
def age_heatmap(chain, colorscale, title):
    rows = cohort_history[cohort_history["chain"] == chain]
    if rows.empty:
        print(f"No {chain} cohorts to plot.")
        return
    grid = rows.pivot_table(index="age_weeks", columns="week_index", values="population", aggfunc="sum", fill_value=0.0)
    ages = range(int(rows["age_weeks"].min()), int(rows["age_weeks"].max()) + 1)
    grid = grid.reindex(index=ages, columns=SNAPSHOT_WEEKS, fill_value=0.0)
    fig = go.Figure(
        go.Heatmap(x=SNAPSHOT_DATES, y=grid.index, z=grid.values.round(0), colorscale=colorscale, colorbar=dict(title="Plants"))
    )
    fig.update_layout(title=title, xaxis_title="Date", yaxis_title="age_weeks", template="plotly_white")
    fig.show()


age_heatmap("pre_flowering", "Blues", f"{DATA_DIR.name}: pre-flowering plants by age (weeks since selection/planting)")

In [63]:
age_heatmap("post_flowering", "Oranges", f"{DATA_DIR.name}: bunches on the plant by age (weeks since bagging)")

In [64]:
# The average weekly cohort: the share of it leaving the chain at each age (flowering on the
# pre-flowering chain; harvest plus field losses on the post-flowering chain), from the mean number
# of plants at each age over every snapshot. A cohort of snapshot age a leaves at age a during that week.
def exits_by_age(chain):
    rows = cohort_history[cohort_history["chain"] == chain]
    if rows.empty:
        return pd.Series(dtype=float)
    at_age = rows.groupby("age_weeks")["population"].sum() / len(SNAPSHOT_WEEKS)
    at_age = at_age.reindex(range(int(at_age.index.min()), int(at_age.index.max()) + 2), fill_value=0.0)
    return ((at_age - at_age.shift(-1)) / at_age.iloc[0]).dropna()


for chain, verb in [("pre_flowering", "flowered"), ("post_flowering", "harvested or lost")]:
    cumulative = exits_by_age(chain).cumsum()
    ages = [next((int(age) for age, share in cumulative.items() if share >= q), None) for q in (0.1, 0.5, 0.9)]
    print(f"{chain}: 10% / 50% / 90% of an average weekly cohort {verb} by age {ages[0]} / {ages[1]} / {ages[2]} weeks")

cut_rows = production_df[production_df["cut"] > 0]
recorded_age_at_cut = cut_rows.groupby("age_at_cut")["cut"].sum() / cut_rows["cut"].sum()
recorded_age_at_cut.index = recorded_age_at_cut.index.astype(int)
display(
    pd.DataFrame(
        {
            "model: share harvested or lost at this age": exits_by_age("post_flowering").round(3),
            "recorded: share of bunches cut at this age": recorded_age_at_cut.round(3),
        }
    )
    .rename_axis("weeks since bagging")
    .T
)

pre_flowering: 10% / 50% / 90% of an average weekly cohort flowered by age 33 / 36 / 43 weeks
post_flowering: 10% / 50% / 90% of an average weekly cohort harvested or lost by age 11 / 13 / 14 weeks


weeks since bagging,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
model: share harvested or lost at this age,0.002,0.002,0.002,0.002,0.001,0.001,0.001,0.004,0.002,0.002,0.095,0.385,0.383,0.110,0.008
recorded: share of bunches cut at this age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.096,0.393,0.390,0.113,0.008


On `data/synthetic` nothing grows: over the 361 weekly snapshots the pre-flowering chain stays
between 317,800 and 329,185 plants and the bunches on the plant between 94,521 and 119,313, because
every mother keeps exactly one follower (3,125,927 followers seeded against 3,127,844 plants
flowering), so the number of production units is constant. In the heatmaps each diagonal stripe is
one week's cohort ageing one week per week: followers flower mostly 33-43 weeks after selection,
and bunches are harvested 11-14 weeks after bagging, the model's share leaving at each of those
ages within 0.01 of the recorded share cut at that age (the small model shares before 11 weeks
are field losses). The upper edge of both heatmaps (and the oldest bands of
the stacked areas) moves with the season: development runs on degree-days, so cohorts flower and
are harvested younger in warm spells.

## Takeaways

- **One engine, one call.** `SimbaPopEngine` is the only entry point -- for a one-row seed, a field
  census or a resumed snapshot, with or without real field records.
- **One reader per file type, one standardized schema each.** `WeatherFileReader`,
  `RibbonProductionReader`, `SuckerSelectionReader`, `PopulationInventoryReader` -- every one
  discards columns outside its own schema and never guesses at a vendor-specific format (see
  [adapt_legacy_exports.py](utils/adapt_legacy_exports.py) for converting legacy exports, and
  [build_farm_inputs.py](utils/build_farm_inputs.py) for building every csv of a farm directory, initial
  population included).
- **Real data drives the model; missing data falls back, never silently.** Every substitution the
  engine makes is in `.notices`, with a message that names exactly what was missing and what was
  used instead.
- **One follower per mother.** Under the default `rate_basis = "cohort"` every plant flowers once,
  is harvested once and keeps one follower, so the farm's age structure stays stable instead of
  growing.